# Lab Day 2: backbone, công thức huấn luyện và suy luận trên DeepWeeds (Lương Sỹ Khánh, 02715)

Notebook này gom đúng các lệnh đã dùng để tạo mọi kết quả trong `report.md`, theo thứ tự chạy. Môi trường: **Kaggle, GPU T4, Internet bật**.
Trong bài thực tế, B01–B06 và T01 chạy trên Google Colab (cùng T4), các bước còn lại trên Kaggle thành nhiều notebook; chạy lại toàn bộ ở đây cho ra cùng cấu hình
(không tái lập từng bit vì `cudnn.benchmark=True`). Thời gian ước tính trên T4: B 70 phút, T00×3 + T02–T11 khoảng 2,5 giờ, F01 seed 1–2 khoảng 25 phút, Bước 3 và test khoảng 30 phút.

Quy tắc: chọn mọi thứ trên **val**; test chỉ mở trong `final_test.py`, đúng một lần mỗi seed (script từ chối ghi đè).
Notebook Kaggle công khai đã dùng: https://www.kaggle.com/code/kascnte/notebook-3b và https://www.kaggle.com/code/kascnte/notebookc-3c

## 0. Cài đặt và dữ liệu

In [ ]:
!pip -q install timm openpyxl gdown
%cd /kaggle/working
!git clone -q https://github.com/kascenite/K4-Track4-Day2-LuongSyKhanh-02715 repo
!mkdir -p /tmp/data/labels && for n in labels train_subset0 val_subset0 test_subset0; do wget -q -O /tmp/data/labels/$n.csv https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels/$n.csv; done
# images.zip: nguồn Zenodo https://zenodo.org/records/7939060/files/images.zip?download=1 (MD5 b7b30f96d466fba86016aa5a26606e0f)
!wget -q -O /tmp/images.zip "https://zenodo.org/records/7939060/files/images.zip?download=1" && md5sum /tmp/images.zip && unzip -q -j -o /tmp/images.zip '*.jpg' -d /tmp/data && rm /tmp/images.zip && ls /tmp/data | grep -c jpg

In [ ]:
CODE = "/kaggle/working/repo/submissions/02715_luong_sy_khanh/code"
OUT = "/kaggle/working/out"
LAB = "/tmp/data/labels"
COMMON = f"images_dir=/tmp/data labels_dir={LAB} out_dir={OUT}/runs pred_dir={OUT}/predictions curves_dir={OUT}/curves ckpt_dir=/tmp/ckpt"
import os; os.makedirs(f"{OUT}/logs", exist_ok=True)

## 1. EDA, kiểm tra pipeline, unit test

In [ ]:
!cd {CODE} && python eda.py /tmp/data {LAB} {OUT}/eda 2>&1 | tee {OUT}/logs/eda.log
!cd {CODE} && python smoke.py /tmp/data {LAB} 2>&1 | tee {OUT}/logs/smoke.log
!cd {CODE} && python -m unittest test_components -v 2>&1 | tee {OUT}/logs/unittests.log

## 2. Bước 1: so sánh 6 backbone (công thức nền, seed 0). `jobs_B.txt` nằm trong thư mục code

In [ ]:
!cd {CODE} && python run_queue.py --jobs jobs_B.txt --common {COMMON} 2>&1 | tee {OUT}/logs/queue_B.log

## 3. Bước 2: nền T00 (3 seed) và ablation T01–T11, mỗi ablation chỉ khác nền một yếu tố (ConvNeXt-T)

In [ ]:
%%writefile /kaggle/working/jobs_T.txt
exp_id=T00 seed=0 backbone=convnext_tiny desc=baseline
exp_id=T00 seed=1 backbone=convnext_tiny desc=baseline
exp_id=T00 seed=2 backbone=convnext_tiny desc=baseline
exp_id=T01 backbone=convnext_tiny init=frozen desc=frozen_head
exp_id=T02 backbone=convnext_tiny crop_min=0.35 desc=crop035
exp_id=T03 backbone=convnext_tiny aug=trivial desc=trivialaug
exp_id=T04 backbone=convnext_tiny mix=cutmix desc=cutmix
exp_id=T05 backbone=convnext_tiny loss=ls label_smoothing=0.1 desc=label_smoothing
exp_id=T06 backbone=convnext_tiny loss=focal focal_gamma=2.0 desc=focal
exp_id=T07 backbone=convnext_tiny loss=ce_weighted class_weight_beta=0.0 desc=class_weighted_ce
exp_id=T08 backbone=convnext_tiny sampler=balanced desc=balanced_sampler
exp_id=T09 backbone=convnext_tiny lr_backbone=0.0002 lr_head=0.002 desc=lr_2x
exp_id=T10 backbone=convnext_tiny ema_decay=0.998 desc=ema
exp_id=T11 backbone=convnext_tiny epochs=20 desc=epochs20

In [ ]:
!cd {CODE} && python run_queue.py --jobs /kaggle/working/jobs_T.txt --common {COMMON} 2>&1 | tee {OUT}/logs/queue_T.log

## 4. Chung kết F01: công thức T03 (TrivialAugment), seed 1 và 2. Seed 0 là checkpoint của T03 seed 0

In [ ]:
%%writefile /kaggle/working/jobs_F.txt
exp_id=F01 seed=1 backbone=convnext_tiny aug=trivial desc=final_trivialaug
exp_id=F01 seed=2 backbone=convnext_tiny aug=trivial desc=final_trivialaug

In [ ]:
!cd {CODE} && python run_queue.py --jobs /kaggle/working/jobs_F.txt --common {COMMON} 2>&1 | tee {OUT}/logs/train_F01.log

## 5. Bước 3: suy luận trên VAL (GPU rảnh) và độ trễ các backbone

In [ ]:
!cd {CODE} && python step3.py --exp T03 --seed 0 --ensemble T03 T02 T05 --cnn 1 --common {COMMON} 2>&1 | tee {OUT}/logs/step3.log
!cd {CODE} && python final_test.py --train_exp T03 --name F01 --seeds 0 --method flip --space prob --temp 1 --dry_run --common {COMMON} 2>&1 | tee {OUT}/logs/dryrun.log
!cd {CODE} && python lat_backbones.py {OUT}/latency_backbones.csv resnet50 resnext50_32x4d convnext_tiny deit_small_patch16_224 efficientnet_b0 mobilenetv3_large_100 2>&1 | tee {OUT}/logs/lat_backbones.log

## 6. Bước 4: chạy TEST đúng một lần mỗi seed

Phương pháp suy luận chọn trên val theo quy tắc đặt trước: chọn phương pháp rẻ nhất, trừ khi phương pháp đắt hơn hơn ≥ 0,001 macro-F1 val. Kết quả: `res256` (resize cả ảnh 256, không crop), gộp xác suất, temperature scaling.
**Đừng chạy lại các ô này sau khi đã có file `*_test.csv`.**

In [ ]:
!cd {CODE} && python final_test.py --train_exp T00 --name T00 --seeds 0 1 2 --method 1view --space prob --temp 0 --common {COMMON} 2>&1 | tee {OUT}/logs/final_T00.log
!cd {CODE} && python final_test.py --train_exp T03 --name F01 --seeds 0 --method res256 --space prob --temp 1 --common {COMMON} 2>&1 | tee {OUT}/logs/final_F01_s0.log
!cd {CODE} && python final_test.py --train_exp F01 --name F01 --seeds 1 2 --method res256 --space prob --temp 1 --common {COMMON} 2>&1 | tee {OUT}/logs/final_F01_s12.log

In [ ]:
EV = "/kaggle/working/repo/eval.py"; P = f"{OUT}/predictions"; T = f"--test-csv {LAB}/test_subset0.csv --labels {LAB}/labels.csv"
!python {EV} score --pred "{P}/T00_seed*_test.csv" {T} --tag T00 --out {OUT}/eval_out 2>&1 | tee {OUT}/logs/score_T00.log
!python {EV} score --pred "{P}/F01_seed*_test.csv" {T} --tag F01 --out {OUT}/eval_out 2>&1 | tee {OUT}/logs/score_F01.log
!python {EV} grade --final "{P}/F01_seed*_test.csv" --baseline "{P}/T00_seed*_test.csv" --uncal "{P}/F01uncal_seed*_test.csv" --final-val "{P}/F01_seed*_val.csv" --latency-p95-ms 10.14 --val-csv {LAB}/val_subset0.csv {T} --out {OUT}/eval_out 2>&1 | tee {OUT}/logs/grade.log

## 7. Bảng tổng hợp, phân tích lỗi, lệch phân phối, DINOv2 (điểm thưởng)

In [ ]:
!cd {CODE} && python make_results.py --root {OUT} --out {OUT}/results.xlsx --final F01 --baseline T00 --seeds 0 1 2 2>&1 | tee {OUT}/logs/make_results.log
!cd {CODE} && PYTHONPATH=/kaggle/working/repo python analysis3c.py --exp T03 --seed 0 --ckpt_dir /tmp/ckpt --out_dir {OUT}/runs --pred_dir {OUT}/predictions --images /tmp/data --labels {LAB} --out {OUT}/analysis 2>&1 | tee {OUT}/logs/analysis3c.log
!cd {CODE} && PYTHONPATH=/kaggle/working/repo python dino_probe.py /tmp/data {LAB} {OUT}/analysis/dino_probe.json 2>&1 | tee {OUT}/logs/dino_probe.log